<a href="https://colab.research.google.com/github/thanhtam-04/NAU-CS077-012026-lab10032026-thanhtam-04/blob/main/ThanhTam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# --- BƯỚC 1: KẾT NỐI GOOGLE DRIVE ---
from google.colab import drive

# Gắn Google Drive vào thư mục /content/drive
drive.mount('/content/drive')

print("✅ Đã kết nối Google Drive thành công!")

# --- BƯỚC 2: CÀI ĐẶT PYSPARK & KHỞI TẠO (Như các bài trước) ---
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install -q pyspark==3.5.1

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, year, month, sum

spark = SparkSession.builder.appName("Drive_DataLake").getOrCreate()
print("✅ Hệ thống Spark đã sẵn sàng!")

Mounted at /content/drive
✅ Đã kết nối Google Drive thành công!
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.
✅ Hệ thống Spark đã sẵn sàng!


In [3]:
import os

# 1. Định nghĩa đường dẫn Data Lake trên Drive
datalake_path = '/content/drive/MyDrive/DataLake_Lab'
raw_zone = f'{datalake_path}/Raw_Zone'
gold_zone = f'{datalake_path}/Gold_Zone'

# 2. Tạo các thư mục (nếu chưa có)
os.makedirs(raw_zone, exist_ok=True)
os.makedirs(gold_zone, exist_ok=True)
print(f"📂 Đã tạo cấu trúc thư mục Data Lake tại: {datalake_path}")

# 3. Tải file CSV thô trực tiếp VÀO GOOGLE DRIVE
csv_file_path = f"{raw_zone}/sales_data.csv"

# Tải file retail data quen thuộc vào thư mục trên Drive
!wget -q https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv -O "{csv_file_path}"
print(f"✅ Đã tải dữ liệu thô (CSV) lưu vĩnh viễn vào Google Drive: {csv_file_path}")

📂 Đã tạo cấu trúc thư mục Data Lake tại: /content/drive/MyDrive/DataLake_Lab
✅ Đã tải dữ liệu thô (CSV) lưu vĩnh viễn vào Google Drive: /content/drive/MyDrive/DataLake_Lab/Raw_Zone/sales_data.csv


In [4]:
print("⏳ Đang đọc dữ liệu thô từ Google Drive...")
# Đọc dữ liệu từ Drive
df_raw = spark.read.csv(csv_file_path, header=True, inferSchema=True)

# Làm sạch (Bỏ Null, số lượng âm)
df_clean = df_raw.filter(
    col("CustomerID").isNotNull() &
    (col("Quantity") > 0)
)

# Biến đổi: Ép kiểu thời gian và tạo cột Năm, Tháng để lát nữa phân vùng
df_transformed = df_clean.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm")) \
                         .withColumn("Year", year(col("InvoiceDate"))) \
                         .withColumn("Month", month(col("InvoiceDate"))) \
                         .withColumn("TotalAmount", col("Quantity") * col("UnitPrice"))

print("✅ Đã xử lý xong dữ liệu trong RAM. Xem trước 3 dòng:")
df_transformed.select("InvoiceNo", "TotalAmount", "Year", "Month", "Country").show(3)

⏳ Đang đọc dữ liệu thô từ Google Drive...
✅ Đã xử lý xong dữ liệu trong RAM. Xem trước 3 dòng:
+---------+------------------+----+-----+--------------+
|InvoiceNo|       TotalAmount|Year|Month|       Country|
+---------+------------------+----+-----+--------------+
|   536365|15.299999999999999|2010|   12|United Kingdom|
|   536365|             20.34|2010|   12|United Kingdom|
|   536365|              22.0|2010|   12|United Kingdom|
+---------+------------------+----+-----+--------------+
only showing top 3 rows



In [5]:
# --- TẠO CELL MỚI VÀ CHẠY ĐOẠN CODE NÀY ---

print("⏳ Đang ghi dữ liệu chuẩn ra Google Drive (có thể mất 1-2 phút)...")

# Đường dẫn thư mục đích trên Drive
parquet_output_path = f"{gold_zone}/sales_parquet"

# Lệnh ghi thần thánh:
df_transformed.write \
              .partitionBy("Year", "Month") \
              .mode("overwrite") \
              .parquet(parquet_output_path)

print(f"🎉 HOÀN TẤT! Dữ liệu đã được lưu vĩnh viễn tại thư mục: {parquet_output_path}")

⏳ Đang ghi dữ liệu chuẩn ra Google Drive (có thể mất 1-2 phút)...
🎉 HOÀN TẤT! Dữ liệu đã được lưu vĩnh viễn tại thư mục: /content/drive/MyDrive/DataLake_Lab/Gold_Zone/sales_parquet


In [6]:
# 1. Liệt kê các thư mục phân vùng được tạo tự động trên Drive
print("📂 Cấu trúc thư mục Parquet trên Drive của bạn:")
!ls -l "{gold_zone}/sales_parquet"

# 2. Thử nghiệm tốc độ đọc dữ liệu
print("\n🚀 Đọc thử đúng dữ liệu của Tháng 12 năm 2011:")

# Nhờ có Partition, Spark không cần đọc cả file lớn,
# nó nhảy thẳng vào thư mục Year=2011/Month=12 để lấy dữ liệu!
df_december = spark.read.parquet(f"{gold_zone}/sales_parquet/Year=2011/Month=12")

print(f"👉 Số lượng đơn hàng trong tháng 12/2011: {df_december.count()}")
df_december.show(3)

📂 Cấu trúc thư mục Parquet trên Drive của bạn:
total 8
-rw-------  1 root root    0 Mar 10 06:59  _SUCCESS
drwx------  3 root root 4096 Mar 10 06:59 'Year=2010'
drwx------ 14 root root 4096 Mar 10 06:59 'Year=2011'

🚀 Đọc thử đúng dữ liệu của Tháng 12 năm 2011:
👉 Số lượng đơn hàng trong tháng 12/2011: 17304
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|       TotalAmount|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|   579899|    23301|GARDENERS KNEELIN...|      24|2011-12-01 08:33:00|     1.65|     15687|United Kingdom|39.599999999999994|
|   579899|    22623|BOX OF VINTAGE JI...|       3|2011-12-01 08:33:00|     5.95|     15687|United Kingdom|             17.85|
|   579899|    20970|PINK FLORAL FELTC...|       4|2011-